# Neo4j second vector baseline

Node-embedding retrieval baseline with evaluation against `annotaion_q_graph` gold in canonical triple space.

Output CSV: `qnum,f1`.


In [48]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_API_KEY = os.getenv("GPT_API_KEY")

print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))

Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [59]:
# 1) Imports and config

import os
import re
import json
import time
from pathlib import Path
from statistics import mean, median

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pandas as pd
from neo4j import GraphDatabase

BASE_DIR = Path('/Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф')
ANNOT_QG_DIR = BASE_DIR / 'annotaion_q_graph'
OUT_CSV_PATH = BASE_DIR / 'retrieval_outputs' / 'node_embedding_baseline_f1.csv'

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '12345678')

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY') or os.getenv('GPT_API_KEY')

QNUMS = [i for i in range (1, 102)]

NODE_EMB_MODEL = os.getenv('NODE_EMB_MODEL', 'text-embedding-3-small')
QUERY_EMB_MODEL = os.getenv('QUERY_EMB_MODEL', 'text-embedding-3-small')
EMB_DIM = int(os.getenv('EMB_DIM', '1536'))

TOP_N_NODES = int(os.getenv('TOP_N_NODES', '25'))
HOPS = int(os.getenv('HOPS', '2'))
MAX_TRIPLES = int(os.getenv('MAX_TRIPLES', '5000'))
USE_CANONICAL_EVAL = True

TARGET_LABELS = [
    'Encounter', 'DocumentCheck', 'DocumentInstance', 'Questioning',
    'BiometricCheck', 'City', 'Airport', 'Country'
]

EMB_BATCH_SIZE = int(os.getenv('EMB_BATCH_SIZE', '16'))
WRITE_BATCH_SIZE = int(os.getenv('WRITE_BATCH_SIZE', '200'))
SKIP_EXISTING_EMBEDDINGS = True

META_KEYS = [
    'intents', 'mode', 'conditional', 'scope', 'exclude_scope',
    'doc_types', 'topic_keys', 'biometric_modalities', 'aggregate'
]

print('ANNOT_QG_DIR =', ANNOT_QG_DIR)
print('OUT_CSV_PATH =', OUT_CSV_PATH)
print('QNUMS =', QNUMS)


ANNOT_QG_DIR = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/annotaion_q_graph
OUT_CSV_PATH = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/node_embedding_baseline_f1.csv
QNUMS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101]


In [60]:
# 2) Core clients and low-level helpers

def require_env():
    if not OPENAI_API_KEY:
        raise ValueError('OPENAI_API_KEY or GPT_API_KEY is not set')
    if not NEO4J_PASSWORD:
        raise ValueError('NEO4J_PASSWORD is not set')


def get_driver():
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))


def _requests_session():
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        backoff_factor=1.0,
        status_forcelist=[408, 409, 425, 429, 500, 502, 503, 504],
        allowed_methods=frozenset(['POST']),
        raise_on_status=False,
    )
    sess = requests.Session()
    adapter = HTTPAdapter(max_retries=retry, pool_connections=20, pool_maxsize=20)
    sess.mount('https://', adapter)
    sess.mount('http://', adapter)
    return sess


def neo4j_query(driver, query, params=None):
    with driver.session() as session:
        res = session.run(query, params or {})
        return [dict(r) for r in res]


def embed_texts(texts, model):
    if not texts:
        return []

    all_vectors = []
    session = _requests_session()
    try:
        for batch in chunks(texts, EMB_BATCH_SIZE):
            payload = {
                'model': model,
                'input': batch,
            }

            last_err = None
            for attempt in range(1, 7):
                try:
                    resp = session.post(
                        'https://api.openai.com/v1/embeddings',
                        headers={
                            'Authorization': f'Bearer {OPENAI_API_KEY}',
                            'Content-Type': 'application/json',
                        },
                        json=payload,
                        timeout=(20, 180),
                    )
                    if resp.ok:
                        data = resp.json().get('data', [])
                        data = sorted(data, key=lambda x: x['index'])
                        vecs = [d['embedding'] for d in data]
                        if len(vecs) != len(batch):
                            raise RuntimeError(f'Embeddings size mismatch: got {len(vecs)}, expected {len(batch)}')
                        all_vectors.extend(vecs)
                        last_err = None
                        break

                    body = ''
                    try:
                        body = resp.text[:800]
                    except Exception:
                        body = '<no-body>'
                    if resp.status_code in {408, 409, 425, 429, 500, 502, 503, 504}:
                        last_err = RuntimeError(f'Retriable API error {resp.status_code}: {body}')
                    else:
                        raise RuntimeError(f'Embeddings API error {resp.status_code}: {body}')

                except requests.exceptions.SSLError as e:
                    last_err = e
                except requests.exceptions.ConnectionError as e:
                    last_err = e
                except requests.exceptions.Timeout as e:
                    last_err = e

                if attempt < 6:
                    sleep_sec = min(20, (2 ** (attempt - 1)))
                    time.sleep(sleep_sec)
                else:
                    raise RuntimeError(f'Embeddings failed after retries: {last_err}')

            if last_err is not None:
                raise RuntimeError(f'Embeddings failed for batch: {last_err}')
    finally:
        session.close()

    return all_vectors


def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def _extract_post_id_from_key(value: str):
    if not value or not isinstance(value, str):
        return None
    m = re.search(r'encounter_(\d+)', value)
    if m:
        return m.group(1)
    m = re.search(r'questioning_encounter_(\d+)', value)
    if m:
        return m.group(1)
    m = re.search(r'doccheck_encounter_(\d+)', value)
    if m:
        return m.group(1)
    m = re.search(r'documentinstance_encounter_(\d+)', value)
    if m:
        return m.group(1)
    return None


In [61]:
# 3) Gold loading and canonicalization (compatible with neo4j_sem_rag)

def annotation_files_for_qnum(qnum: int):
    qdir = ANNOT_QG_DIR / f'q_{qnum}'
    if not qdir.exists():
        raise FileNotFoundError(f'Question directory not found: {qdir}')
    files = sorted(qdir.glob('*/*.json'))
    ann = [f for f in files if f.name.startswith('annotation_') and f.suffix == '.json']
    return ann


def get_question_text(qnum: int) -> str:
    files = annotation_files_for_qnum(qnum)
    if not files:
        raise ValueError(f'No annotation files found for q_{qnum}')
    data = json.loads(files[0].read_text(encoding='utf-8'))
    question = data.get('question')
    if not question:
        raise ValueError(f'Question text not found in {files[0]}')
    return question


def load_gold(qnum: int):
    qdir = ANNOT_QG_DIR / f'q_{qnum}'
    triples = []
    metas = []
    for f in annotation_files_for_qnum(qnum):
        data = json.loads(f.read_text(encoding='utf-8'))
        for t in data.get('triples', []):
            s = t.get('s', {})
            o = t.get('o', {})
            triples.append((s.get('label'), s.get('key'), t.get('p'), o.get('label'), o.get('key')))
        meta = {k: data.get(k) for k in META_KEYS if k in data}
        if meta:
            metas.append(meta)
    meta_ref = metas[0] if metas else {}
    meta_consistent = all(m == meta_ref for m in metas) if metas else True

    agg_ref = None
    agg_path = qdir / 'aggregate_output.json'
    if agg_path.exists():
        agg_ref = json.loads(agg_path.read_text(encoding='utf-8'))

    return set(triples), meta_ref, meta_consistent, agg_ref


def _canonical_post_key_from_annotation(data: dict):
    for t in data.get('triples', []):
        for side in ['s', 'o']:
            key = (t.get(side) or {}).get('key')
            post_id = _extract_post_id_from_key(key)
            if post_id:
                return f'post_{post_id}'
    return None


def load_gold_canonical(qnum: int):
    canonical = set()
    encounters = {}
    _, meta_ref, _, agg_ref = load_gold(qnum)

    for f in annotation_files_for_qnum(qnum):
        data = json.loads(f.read_text(encoding='utf-8'))
        post_key = _canonical_post_key_from_annotation(data)
        if not post_key:
            continue
        enc = encounters.setdefault(post_key, {
            'post_key': post_key,
            'atCountry': set(),
            'atCity': set(),
            'atAirport': set(),
            'document_types': set(),
            'topic_keys': set(),
            'biometric_modalities': set(),
        })
        for t in data.get('triples', []):
            s = t.get('s', {})
            o = t.get('o', {})
            p = t.get('p')
            if s.get('label') == 'Encounter' and p in {'atCountry', 'atCity', 'atAirport'}:
                canonical.add((s.get('label'), post_key, p, o.get('label'), o.get('key')))
                if p == 'atCountry':
                    enc['atCountry'].add(o.get('key'))
                elif p == 'atCity':
                    enc['atCity'].add(o.get('key'))
                elif p == 'atAirport':
                    enc['atAirport'].add(o.get('key'))
            elif p == 'documentType' and o.get('key'):
                canonical.add(('Encounter', post_key, 'documentType', 'Literal', o.get('key')))
                enc['document_types'].add(o.get('key'))
            elif p == 'topicKey' and o.get('key'):
                canonical.add(('Encounter', post_key, 'topicKey', 'Literal', o.get('key')))
                enc['topic_keys'].add(o.get('key'))
            elif p == 'biometricModality' and o.get('key'):
                canonical.add(('Encounter', post_key, 'biometricModality', 'Literal', o.get('key')))
                enc['biometric_modalities'].add(o.get('key'))

    return canonical, encounters, meta_ref, agg_ref


In [62]:
# 4) Node text representation and embedding build

def _vector_index_name(label: str) -> str:
    return f'vec_{label.lower()}_embedding'


def ensure_vector_indexes(driver, emb_dim: int = EMB_DIM):
    for label in TARGET_LABELS:
        idx_name = _vector_index_name(label)
        q = f"""
        CREATE VECTOR INDEX {idx_name} IF NOT EXISTS
        FOR (n:`{label}`) ON (n.embedding)
        OPTIONS {{indexConfig: {{`vector.dimensions`: {emb_dim}, `vector.similarity_function`: 'cosine'}}}}
        """
        neo4j_query(driver, q)


def fetch_nodes_for_embedding(driver, limit=None, skip_existing: bool = True):
    where_emb = 'AND n.embedding IS NULL' if skip_existing else ''
    lim = f'LIMIT {int(limit)}' if limit else ''
    q = f"""
    MATCH (n)
    WHERE any(l IN labels(n) WHERE l IN $labels)
      AND n.key IS NOT NULL
      {where_emb}
    RETURN id(n) AS nid, labels(n) AS labels, properties(n) AS props
    {lim}
    """
    return neo4j_query(driver, q, {'labels': TARGET_LABELS})


def fetch_node_context(driver, nid: int):
    q = """
    MATCH (n) WHERE id(n) = $nid
    OPTIONAL MATCH (n)-[r]-(m)
    RETURN collect(DISTINCT type(r))[0..20] AS rel_types,
           collect(DISTINCT coalesce(head(labels(m)), '') + ':' + coalesce(m.key, ''))[0..20] AS neighbors
    """
    rows = neo4j_query(driver, q, {'nid': nid})
    if not rows:
        return {'rel_types': [], 'neighbors': []}
    return rows[0]


def node_to_text(node_row, ctx):
    labels = ','.join(node_row.get('labels') or [])
    props = dict(node_row.get('props') or {})
    key = props.get('key', '')
    props.pop('embedding', None)

    scalar_props = []
    for k, v in sorted(props.items()):
        if k == 'key':
            continue
        if isinstance(v, (str, int, float, bool)):
            scalar_props.append(f'{k}={v}')

    rel_types = ', '.join(x for x in (ctx.get('rel_types') or []) if x)
    neighbors = ', '.join(x for x in (ctx.get('neighbors') or []) if x)

    return (
        f'labels: {labels}\n'
        f'key: {key}\n'
        f'properties: {'; '.join(scalar_props)}\n'
        f'relations: {rel_types}\n'
        f'neighbors: {neighbors}'
    )


def write_node_embeddings(driver, rows):
    if not rows:
        return
    q = """
    UNWIND $rows AS row
    MATCH (n) WHERE id(n) = row.nid
    SET n.embedding = row.embedding
    """
    neo4j_query(driver, q, {'rows': rows})


def build_node_embeddings(limit=None, overwrite=False):
    require_env()
    with get_driver() as driver:
        ensure_vector_indexes(driver, emb_dim=EMB_DIM)
        nodes = fetch_nodes_for_embedding(driver, limit=limit, skip_existing=(SKIP_EXISTING_EMBEDDINGS and not overwrite))
        print('nodes_to_embed =', len(nodes))

        prepared = []
        for row in nodes:
            ctx = fetch_node_context(driver, row['nid'])
            prepared.append({
                'nid': row['nid'],
                'text': node_to_text(row, ctx),
            })

        t0 = time.time()
        written = 0
        for text_batch in chunks(prepared, EMB_BATCH_SIZE):
            vectors = embed_texts([x['text'] for x in text_batch], model=NODE_EMB_MODEL)
            out_rows = [
                {'nid': text_batch[i]['nid'], 'embedding': vectors[i]}
                for i in range(len(text_batch))
            ]
            for wbatch in chunks(out_rows, WRITE_BATCH_SIZE):
                write_node_embeddings(driver, wbatch)
                written += len(wbatch)

        print('embedded_written =', written)
        print('elapsed_sec =', round(time.time() - t0, 2))


In [63]:
# 5) Vector retrieval + hop expansion + canonical triple construction

def get_existing_vector_indexes(driver):
    q = """
    SHOW INDEXES
    YIELD name, type
    WHERE type = 'VECTOR'
    RETURN collect(name) AS names
    """
    rows = neo4j_query(driver, q)
    names = set(rows[0]['names']) if rows else set()
    return names


def query_seed_nodes(driver, question: str, top_n: int = TOP_N_NODES):
    qvec = embed_texts([question], model=QUERY_EMB_MODEL)[0]
    existing = get_existing_vector_indexes(driver)

    scored = {}
    k_each = max(top_n, 1)
    for label in TARGET_LABELS:
        idx = _vector_index_name(label)
        if idx not in existing:
            continue
        q = """
        CALL db.index.vector.queryNodes($index_name, $k, $embedding)
        YIELD node, score
        RETURN id(node) AS nid, labels(node) AS labels, node.key AS key, score
        """
        rows = neo4j_query(driver, q, {'index_name': idx, 'k': k_each, 'embedding': qvec})
        for r in rows:
            nid = r['nid']
            if nid not in scored or r['score'] > scored[nid]['score']:
                scored[nid] = r

    ranked = sorted(scored.values(), key=lambda x: -float(x['score']))[:top_n]
    return ranked


def expand_node_ids(driver, seed_ids, hops: int = HOPS):
    if not seed_ids:
        return []
    hops = max(0, int(hops))
    q = f"""
    MATCH (s) WHERE id(s) IN $seed_ids
    MATCH p=(s)-[*0..{hops}]-(n)
    RETURN collect(DISTINCT id(n)) AS node_ids
    """
    rows = neo4j_query(driver, q, {'seed_ids': seed_ids})
    if not rows:
        return []
    return rows[0].get('node_ids') or []


def fetch_encounter_facts_from_node_ids(driver, node_ids):
    if not node_ids:
        return {}
    q = """
    MATCH (e:Encounter)
    WHERE id(e) IN $node_ids

    OPTIONAL MATCH (e)-[:atCountry]->(co:Country)
    OPTIONAL MATCH (e)-[:atCity]->(ci:City)
    OPTIONAL MATCH (e)-[:atAirport]->(ap:Airport)

    OPTIONAL MATCH (e)-[:hasDocumentCheck]->(:DocumentCheck)-[:requestedDocument]->(di:DocumentInstance)
    OPTIONAL MATCH (di)-[:documentType]->(dt)

    OPTIONAL MATCH (e)-[:hasQuestioning]->(q:Questioning)
    OPTIONAL MATCH (q)-[:topicKey]->(tk)

    OPTIONAL MATCH (e)-[:hasBiometricCheck]->(b:BiometricCheck)
    OPTIONAL MATCH (b)-[:biometricModality]->(bm)

    RETURN e.key AS encounter_key,
           collect(DISTINCT co.key) AS countries,
           collect(DISTINCT ci.key) AS cities,
           collect(DISTINCT ap.key) AS airports,
           collect(DISTINCT coalesce(dt.key, di.documentType)) AS docs,
           collect(DISTINCT coalesce(tk.key, q.topicKey)) AS topics,
           collect(DISTINCT coalesce(bm.key, b.biometricModality)) AS bios
    """
    rows = neo4j_query(driver, q, {'node_ids': node_ids})

    encounters = {}
    for r in rows:
        ekey = r.get('encounter_key')
        pid = _extract_post_id_from_key(ekey)
        if not pid:
            continue
        post_key = f'post_{pid}'
        encounters[post_key] = {
            'post_key': post_key,
            'atCountry': set(x for x in (r.get('countries') or []) if x),
            'atCity': set(x for x in (r.get('cities') or []) if x),
            'atAirport': set(x for x in (r.get('airports') or []) if x),
            'document_types': set(x for x in (r.get('docs') or []) if x),
            'topic_keys': set(x for x in (r.get('topics') or []) if x),
            'biometric_modalities': set(x for x in (r.get('bios') or []) if x),
        }
    return encounters


def encounter_rows_to_canonical_triples(encounter_rows):
    triples = set()
    for rec in encounter_rows.values():
        post_key = rec['post_key']
        for x in sorted(rec['atCountry']):
            triples.add(('Encounter', post_key, 'atCountry', 'Country', x))
        for x in sorted(rec['atCity']):
            triples.add(('Encounter', post_key, 'atCity', 'City', x))
        for x in sorted(rec['atAirport']):
            triples.add(('Encounter', post_key, 'atAirport', 'Airport', x))
        for x in sorted(rec['document_types']):
            triples.add(('Encounter', post_key, 'documentType', 'Literal', x))
        for x in sorted(rec['topic_keys']):
            triples.add(('Encounter', post_key, 'topicKey', 'Literal', x))
        for x in sorted(rec['biometric_modalities']):
            triples.add(('Encounter', post_key, 'biometricModality', 'Literal', x))
    return triples


def retrieve_canonical_triples_by_qnum(qnum: int, top_n_nodes: int = TOP_N_NODES, hops: int = HOPS):
    question = get_question_text(qnum)
    with get_driver() as driver:
        seeds = query_seed_nodes(driver, question, top_n=top_n_nodes)
        seed_ids = [r['nid'] for r in seeds]
        expanded_ids = expand_node_ids(driver, seed_ids, hops=hops)
        encounters = fetch_encounter_facts_from_node_ids(driver, expanded_ids)

    triples = encounter_rows_to_canonical_triples(encounters)
    if MAX_TRIPLES and len(triples) > MAX_TRIPLES:
        triples = set(list(sorted(triples))[:MAX_TRIPLES])

    return {
        'question': question,
        'seeds': seeds,
        'seed_ids': seed_ids,
        'expanded_ids': expanded_ids,
        'encounters': encounters,
        'triples': triples,
    }


In [64]:
# 6) Evaluation and CSV report (qnum,f1)

def f1_from_sets(pred_set, gold_set):
    tp = len(pred_set & gold_set)
    if not pred_set and not gold_set:
        return 1.0, 1.0, 1.0
    precision = tp / len(pred_set) if pred_set else 0.0
    recall = tp / len(gold_set) if gold_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


def run_eval_qnum(qnum: int, print_details: bool = True):
    pred = retrieve_canonical_triples_by_qnum(qnum)
    gold_canonical, _, _, _ = load_gold_canonical(qnum)

    precision, recall, f1 = f1_from_sets(pred['triples'], gold_canonical)

    if print_details:
        print('qnum', qnum)
        print('question', pred['question'])
        print('seed_nodes', len(pred['seed_ids']))
        print('expanded_nodes', len(pred['expanded_ids']))
        print('retrieved', len(pred['triples']))
        print('gold', len(gold_canonical))
        print('precision', precision)
        print('recall', recall)
        print('f1', f1)

    return {'qnum': int(qnum), 'f1': float(f1)}


def run_eval_batch(qnums, out_csv: Path = OUT_CSV_PATH, print_details: bool = True):
    rows = []
    for q in qnums:
        rows.append(run_eval_qnum(int(q), print_details=print_details))

    df = pd.DataFrame(rows, columns=['qnum', 'f1']).sort_values('qnum').reset_index(drop=True)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv, index=False, encoding='utf-8')

    print('saved_csv =', out_csv)
    print('n_questions =', len(df))
    if len(df):
        print('mean_f1 =', float(df['f1'].mean()))
        print('median_f1 =', float(df['f1'].median()))

    return df


In [65]:
# 7) Run cells

# One-time indexing/build (can be long):
require_env()
build_node_embeddings(limit=None, overwrite=False)

# Smoke test:
smoke_df = run_eval_batch([1], print_details=True)
smoke_df

# Full run by configured QNUMS:
result_df = run_eval_batch(QNUMS, out_csv=OUT_CSV_PATH, print_details=True)
result_df


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=6, column=12, offset=133>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 133, 'line': 6, 'column': 12}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (n)\n    WHERE any(l IN labels(n) WHERE l IN $labels)\n      AND n.key IS NOT NULL\n      AND n.embedding IS NULL\n    RETURN id(n) AS nid, labels(n) AS labels, properties(n) AS props\n    \n    '


nodes_to_embed = 0
embedded_written = 0
elapsed_sec = 0.0


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=16, offset=111>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 111, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $k, $embedding)\n        YIELD node, score\n        RETURN id(node) AS nid, labels(node) AS labels, node.key AS key, score\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is depr

qnum 1
question Which documents do I need to bring to passport control in Paris?
seed_nodes 25
expanded_nodes 280
retrieved 499
gold 299
precision 0.561122244488978
recall 0.9364548494983278
f1 0.7017543859649124
saved_csv = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/node_embedding_baseline_f1.csv
n_questions = 1
mean_f1 = 0.7017543859649124
median_f1 = 0.7017543859649124


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=16, offset=111>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 111, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $k, $embedding)\n        YIELD node, score\n        RETURN id(node) AS nid, labels(node) AS labels, node.key AS key, score\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is depr

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=16, offset=111>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 111, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $k, $embedding)\n        YIELD node, score\n        RETURN id(node) AS nid, labels(node) AS labels, node.key AS key, score\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is depr

qnum 100
question Which question topics are asked for travelers whose documents include return_ticket in France?
seed_nodes 25
expanded_nodes 354
retrieved 545
gold 79
precision 0.11926605504587157
recall 0.8227848101265823
f1 0.20833333333333331


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=16, offset=111>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 111, 'line': 4, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $k, $embedding)\n        YIELD node, score\n        RETURN id(node) AS nid, labels(node) AS labels, node.key AS key, score\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is depr

qnum 101
question Show document checks and biometric modalities for the same encounters in Germany.
seed_nodes 25
expanded_nodes 385
retrieved 1028
gold 25
precision 0.0048638132295719845
recall 0.2
f1 0.00949667616334283
saved_csv = /Users/i.sharganov/🟩🟩🟩/обучение/МАГА/мага_обучение/3_сем/Диплом/эксперимент/neo4j_граф/retrieval_outputs/node_embedding_baseline_f1.csv
n_questions = 101
mean_f1 = 0.22542728295085093
median_f1 = 0.11527377521613831


,qnum,f1
0,1,0.701754
1,2,0.923547
2,3,0.320700
3,4,0.138016
4,5,0.320988
...,...,...
96,97,0.034755
97,98,0.000000
98,99,0.002209
99,100,0.208333


In [66]:
k

NameError: name 'k' is not defined

In [68]:
result_df.head(100)

,qnum,f1
0,1,0.701754
1,2,0.923547
2,3,0.320700
3,4,0.138016
4,5,0.320988
...,...,...
95,96,0.453689
96,97,0.034755
97,98,0.000000
98,99,0.002209
